In [1]:
# Export random sample of Florida permits for data exploration

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np

rng = np.random.RandomState(42)

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

SUMMARY_FILENAME = "dewey_summary"
SUMMARY_FILEPATH = os.path.join(MY_DATA_PATH, f"{SUMMARY_FILENAME}.parquet")

COLUMNS = [
    'PERMIT_NUMBER', 'JURISDICTION', 'STATE', 'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 'STATUS_NORMALIZED', 'STATUS_ORIGINAL', 'RECORD_TYPE_ORIGINAL', 'RECORD_SUBTYPE_ORIGINAL', 'APN', 'STREET', 'ZIPCODE', 'DATA', 'RESIDENTIAL'
]  # Columns to load from data file

TARGET_RECORDS_PER_CITY = 2000
STATE = "FL"
OUTPUT_FILEPATH = os.path.join(MY_DATA_PATH, "processed_data", "permits_fl_sample.parquet")


In [2]:
# Load the data

summ_df = pd.read_parquet(SUMMARY_FILEPATH)
summ_df = summ_df.groupby(['STATE', 'JURISDICTION']).agg(COUNT = ('COUNT', 'sum')).reset_index()
summ_df = summ_df.loc[summ_df['STATE']==STATE].sort_values(by='COUNT', ascending=False).reset_index(drop=True)


In [3]:
# Retrieve cities data

jurisdictions = summ_df['JURISDICTION'].tolist()
states = summ_df['STATE'].tolist()

df = []
for j, s in zip(jurisdictions, states):
    mydf = du.get_data_for_jurisdiction(j, s, columns=COLUMNS, n_records=TARGET_RECORDS_PER_CITY, rng=rng)
    df.append(mydf)

df = pd.concat(df)

print("\nDone!\n")
print(df["JURISDICTION"].value_counts())


Retrieving data for Jacksonville FL ... 196/196 files ... elapsed time 117.37 seconds              
Retrieving data for Lee County FL ... 138/138 files ... elapsed time 71.76 seconds              
Retrieving data for Sarasota County FL ... 43/43 files ... elapsed time 20.83 seconds              
Retrieving data for Osceola County FL ... 67/67 files ... elapsed time 33.47 seconds              
Retrieving data for Orlando FL ... 12/12 files ... elapsed time 5.13 seconds              
Retrieving data for Charlotte County FL ... 73/73 files ... elapsed time 40.15 seconds              
Retrieving data for Pasco County FL ... 62/62 files ... elapsed time 37.52 seconds              
Retrieving data for Miami-Dade County FL ... 18/18 files ... elapsed time 9.93 seconds              
Retrieving data for Cape Coral FL ... 58/58 files ... elapsed time 37.78 seconds              
Retrieving data for Pinellas County FL ... 38/38 files ... elapsed time 22.97 seconds              
Retrieving data for

In [4]:
# Save data
df.to_parquet(OUTPUT_FILEPATH)
